In [1]:
import pandas as pd

In [2]:
%%capture
!pip install -U dspy pydantic dspy-ai

In [3]:
import dspy

api_key = ""
api_endpoint = "https://<endpoint>.services.ai.azure.com/"

lm = dspy.LM('azure/gpt-4.1', api_key = api_key, api_base=api_endpoint, api_version = '2024-10-21', max_tokens=4000)
dspy.configure(lm=lm)

In [4]:
from typing import List
from typing import Literal
from pydantic import BaseModel

class PatientRecord(dspy.Signature):
    """
    You are an expert in clinical NLP in Spanish. 
    Extract contiguous tokens referring to specific toxic habits, like substance use and abuse, from a patient discharge summary as they appear in the Spanish text.
    Make sure to retrieve all tokens of toxic habits, if any.
    """
    
    patient_discharge_summary: str = dspy.InputField(desc="Patient discharge summary")
    #Tobacco, Cannabis, Alcohol and Drug - type
    tobacco_habits: list[str] = dspy.OutputField(desc="All tobacco tokens that can be extracted from the discharge summary. For example: cigarrillos, tabaco, tabaquismo.")
    cannabis_habits: list[str] = dspy.OutputField(desc="All cannabis tokens that can be extracted from the discharge summary. For example: marihuana, cannabis, hachís.")
    alcohol_habits: list[str] = dspy.OutputField(desc="All alcohol tokens that can be extracted from the discharge summary. For example: enolísmo, alcohólica, cervezas.")
    drug_habits: list[str] = dspy.OutputField(desc="All drug tokens that can be extracted from the discharge summary. For example: MDMA, drogas, otros tóxicos, abstinencia, otras sustancias adictivas.")

term_extractor_patient = dspy.ChainOfThought(PatientRecord)

In [5]:
text = """
Mujer de casi 32 años, natural y residente en la zona, en seguimiento en nuestra UCA de Vinaròs desde los 18 años (en 2005); en terapia conmigo desde 2014 (año de mi incorporación a la plaza).
De acuerdo a las notas de la historia clínica de papel y diferentes documentos consultados para la sesión clínica -a menudo desordenados, informes de distinta procedencia y cotejo con apuntes propios-, la paciente se inició en el consumo de tabaco y alcohol a los 12 años, en el cannabis a los 13 (diario desde los 15), en la cocaína, speed, anfetaminas y éxtasis a los 15 también (ocio de los fines de semana, con alcohol), y años más tarde consumo de heroína -fumada y esnifada, mezclada con otras drogas- a los 26, aprox.
Lena dejó los estudios con 16 años, en 2º de la ESO, optando luego por trabajos muy precarios, erráticos, y sobre todo actividades marginales, entre ellas la prostitución hace 2 o 3 años, en Valencia.
Tiene reconocida una PNC (pensión no contributiva) por discapacidad del 73%, de la cual ella siempre se ha administrado el dinero, y hace un año fue inscrita por los Servicios Sociales de la localidad en un curso remunerado de administrativo, donde las condiciones de asistencia eran relativamente exigentes (horarios madrugadores, puntualidad, presencia, exposiciones), y a Lena le costaba bastante cumplir.
Mostraba esfuerzo, también motivada por el incentivo económico, pero había días que no acudía a clase, y ésta era también una de las razones para asistir con más frecuencia y compromiso a las citas médicas y psicológicas, que justificaban la ausencia de clase ese día.
Posteriormente ingresó voluntariamente en un centro de día para rehabilitación de tóxicos, con horarios poco compatibles con el curso de administrativo, sin perder la plaza gracias a una ILT (baja médica).
Respecto al entorno de Lena, la familia se caracteriza por su carencia de estructura.
Los padres de la paciente eran toxicómanos antes y durante su infancia, y ambos ya están fallecidos.
Ella fue acogida por la abuela materna, que también ha sido la persona que ha criado a una hermana 14 años menor, de un padre diferente (éste se encuentra vivo, también era toxicómano, fue presidiario, en la actualidad visita ocasionalmente a la familia, sobre todo a su hija, la hermana de Lena que ahora tiene 18 años).
Por tanto, desde el nacimiento la tutela de la paciente ha estado con los abuelos maternos, que actualmente cuentan con 68 años la mujer y 64 el marido (este hombre puede no ser el abuelo biológico de Lena, según una referencia de un informe de la historia clínica, y es la persona sobre la que actualmente ella deposita más hostilidad y rechazo).
La abuela es quien siempre se encarga de acompañar a Lena a los dispositivos, la ha rescatado en numerosas ocasiones de sitios hostiles y caóticos, ha supervisado muchas veces tratamientos y medicaciones, gestionado citas y visitas… es la principal figura de apego de la paciente, sin duda.
"""
term_extractor_patient(patient_discharge_summary=text)

Prediction(
    reasoning='The discharge summary provides a detailed history of substance use. The patient started "consumo de tabaco y alcohol" at age 12, "en el cannabis a los 13 (diario desde los 15)", and "en la cocaína, speed, anfetaminas y éxtasis a los 15 también (ocio de los fines de semana, con alcohol)", and later "consumo de heroína -fumada y esnifada, mezclada con otras drogas- a los 26, aprox." These are explicit mentions of tobacco, cannabis, alcohol, and various drugs (cocaína, speed, anfetaminas, éxtasis, heroína, drogas). The summary also mentions "rehabilitación de tóxicos" and "toxicómanos" in reference to the patient and her family, which are relevant drug-related terms.',
    tobacco_habits=['tabaco'],
    cannabis_habits=['cannabis'],
    alcohol_habits=['alcohol'],
    drug_habits=['cocaína', 'speed', 'anfetaminas', 'éxtasis', 'heroína', 'drogas', 'tóxicos', 'toxicómanos']
)

In [6]:
lm.history

[{'prompt': None,
  'messages': [{'role': 'system',
    'content': 'Your input fields are:\n1. `patient_discharge_summary` (str): Patient discharge summary\nYour output fields are:\n1. `reasoning` (str): \n2. `tobacco_habits` (list[str]): All tobacco tokens that can be extracted from the discharge summary. For example: cigarrillos, tabaco, tabaquismo.\n3. `cannabis_habits` (list[str]): All cannabis tokens that can be extracted from the discharge summary. For example: marihuana, cannabis, hachís.\n4. `alcohol_habits` (list[str]): All alcohol tokens that can be extracted from the discharge summary. For example: enolísmo, alcohólica, cervezas.\n5. `drug_habits` (list[str]): All drug tokens that can be extracted from the discharge summary. For example: MDMA, drogas, otros tóxicos, abstinencia, otras sustancias adictivas.\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## patient_discharge_summary ## ]]\n{patient_discharge_summary}\n\n

In [7]:
class SplitExtract(dspy.Module):
    def __init__(self):
        self.term_extractor_patient = dspy.ChainOfThought(PatientRecord)

    def forward(self, patient_discharge_summary: str):
        texts = patient_discharge_summary.split('\n\n')
        tobacco_habits, cannabis_habits, alcohol_habits, drug_habits = [], [], [], []
        for text in texts:
            if len(text.strip()) == 0:
                continue
            prediction = self.term_extractor_patient(patient_discharge_summary=text)
            tobacco_habits.extend(prediction.tobacco_habits)
            cannabis_habits.extend(prediction.cannabis_habits)
            alcohol_habits.extend(prediction.alcohol_habits)
            drug_habits.extend(prediction.drug_habits)

        return dspy.Prediction(tobacco_habits=tobacco_habits, cannabis_habits=cannabis_habits, alcohol_habits=alcohol_habits, drug_habits=drug_habits)

In [8]:
doc_extractor = SplitExtract()
doc_extractor(patient_discharge_summary=text)

Prediction(
    tobacco_habits=['tabaco'],
    cannabis_habits=['cannabis'],
    alcohol_habits=['alcohol'],
    drug_habits=['cocaína', 'speed', 'anfetaminas', 'éxtasis', 'heroína', 'drogas', 'tóxicos', 'toxicómanos']
)

In [9]:
lm.history

[{'prompt': None,
  'messages': [{'role': 'system',
    'content': 'Your input fields are:\n1. `patient_discharge_summary` (str): Patient discharge summary\nYour output fields are:\n1. `reasoning` (str): \n2. `tobacco_habits` (list[str]): All tobacco tokens that can be extracted from the discharge summary. For example: cigarrillos, tabaco, tabaquismo.\n3. `cannabis_habits` (list[str]): All cannabis tokens that can be extracted from the discharge summary. For example: marihuana, cannabis, hachís.\n4. `alcohol_habits` (list[str]): All alcohol tokens that can be extracted from the discharge summary. For example: enolísmo, alcohólica, cervezas.\n5. `drug_habits` (list[str]): All drug tokens that can be extracted from the discharge summary. For example: MDMA, drogas, otros tóxicos, abstinencia, otras sustancias adictivas.\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## patient_discharge_summary ## ]]\n{patient_discharge_summary}\n\n

In [10]:
result_filename = 'pred_dev_gpt41_lists_examples_per_field_temp0_optimizers'

In [11]:
df_records = pd.read_csv('final_train_dataset.tsv', sep='\t')

In [12]:
df_records.head()

,filename,text,trigger_annotations,attr_annotations,is_train
0,32073161_ES,"El 21 de enero de 2020, ingresó en el Hospital...","[{'label': 'Tobacco', 'off0': '569', 'off1': '...","[{'label': 'Duration', 'off0': '577', 'off1': ...",True
1,32277408_ES,﻿Un hombre de 63 años ingresó en el hospital a...,"[{'label': 'Tobacco', 'off0': '274', 'off1': '...",[],True
2,32423911_ES,﻿Un hombre de 36 años llegó al servicio de urg...,"[{'label': 'Tobacco', 'off0': '350', 'off1': '...","[{'label': 'Amount', 'off0': '361', 'off1': '3...",False
3,32426200_ES,"Una mujer de 31 años, por lo demás sana, acudi...","[{'label': 'Alcohol', 'off0': '278', 'off1': '...","[{'label': 'Type', 'off0': '299', 'off1': '318...",True
4,32586958_ES,﻿Hombre de 21 años que acudió al servicio de u...,"[{'label': 'Tobacco', 'off0': '277', 'off1': '...",[],True


In [13]:
from ast import literal_eval
df_records['trigger_annotations'] = df_records['trigger_annotations'].apply(literal_eval)
df_records['attr_annotations'] = df_records['attr_annotations'].apply(literal_eval)

In [14]:
df_records['response'] = ''
df_records['entities'] = ''
#df_records['is_train'] = False

In [15]:
df_records_dev = df_records[~df_records['is_train']]
df_records_dev.shape

(300, 7)

In [16]:
def split_text(filename, summary_text, annotations):
    texts = summary_text.split('\n\n')
    annotations_split = []
    current_start_idx, current_end_idx = 0, 0
    start_idx = []
    for text in texts:
        current_start_idx += summary_text[current_start_idx:].index(text)
        current_end_idx = current_start_idx + len(text)
        start_idx.append(current_start_idx)
        # filter annotations between start/end
        current_annotations = []
        for ann in annotations:
            if int(ann['off0']) >= current_start_idx and int(ann['off1']) <= current_end_idx:
                current_annotations.append({
                    'filename': filename,
                    'mark': 'TOX',
                    'label': ann['label'],
                    'off0': int(ann['off0']) - current_start_idx,
                    'off1': int(ann['off1']) - current_start_idx,
                    'span': ann['span'],
                })

        annotations_split.append(current_annotations)
        
    return texts, annotations_split, start_idx

In [17]:
df_train_records = pd.read_csv('final_train_dataset.tsv', sep='\t')
df_train_records['trigger_annotations'] = df_train_records['trigger_annotations'].apply(literal_eval)
df_train_records['attr_annotations'] = df_train_records['attr_annotations'].apply(literal_eval)
df_train_records.head()

,filename,text,trigger_annotations,attr_annotations,is_train
0,32073161_ES,"El 21 de enero de 2020, ingresó en el Hospital...","[{'label': 'Tobacco', 'off0': '569', 'off1': '...","[{'label': 'Duration', 'off0': '577', 'off1': ...",True
1,32277408_ES,﻿Un hombre de 63 años ingresó en el hospital a...,"[{'label': 'Tobacco', 'off0': '274', 'off1': '...",[],True
2,32423911_ES,﻿Un hombre de 36 años llegó al servicio de urg...,"[{'label': 'Tobacco', 'off0': '350', 'off1': '...","[{'label': 'Amount', 'off0': '361', 'off1': '3...",False
3,32426200_ES,"Una mujer de 31 años, por lo demás sana, acudi...","[{'label': 'Alcohol', 'off0': '278', 'off1': '...","[{'label': 'Type', 'off0': '299', 'off1': '318...",True
4,32586958_ES,﻿Hombre de 21 años que acudió al servicio de u...,"[{'label': 'Tobacco', 'off0': '277', 'off1': '...",[],True


In [162]:
dev_dataset_text = []
dev_dataset_text_labels = []
train_dataset_text = []
for index, row in df_train_records.iterrows():
    filename = row['filename']
    #if row['is_train']:
    texts, annotations, start_idx = split_text(filename, row['text'], row['trigger_annotations'])     
    filenames = [f"{filename}_{start_id}" for start_id in start_idx]
    #else:
    #    filenames = [filename]
    #    annotations = [row['trigger_annotations']]
    #    texts = [row['text']]
    #    start_idx = [0]
        
    for (text, current_annotations, filename) in zip(texts, annotations, filenames):        
        items_list_tobacco = [ label['span'] for label in current_annotations if label['label'] == 'Tobacco']
        items_list_cannabis = [ label['span'] for label in current_annotations if label['label'] == 'Cannabis']
        items_list_alcohol = [ label['span'] for label in current_annotations if label['label'] == 'Alcohol']
        items_list_drug = [ label['span'] for label in current_annotations if label['label'] == 'Drug']
    
        example = dspy.Example(patient_discharge_summary=text,
                    filename=filename,
                    annotations=current_annotations,
                    tobacco_habits=items_list_tobacco,
                    cannabis_habits=items_list_cannabis,
                    alcohol_habits=items_list_alcohol,
                    drug_habits=items_list_drug).with_inputs("patient_discharge_summary")
        
        if row['is_train']:        
            if len(current_annotations) > 0:
                train_dataset_text.append(example)
        else:
            dev_dataset_text.append(example)
            if len(current_annotations) > 0:
                dev_dataset_text_labels.append(example)

len(dev_dataset_text)

1735

In [163]:
len(dev_dataset_text_labels)

520

In [164]:
len(train_dataset_text)

1692

In [ ]:
# module to split and combine results

In [165]:
import warnings
def calculate_metrics(gs, pred, subtask=['ner','norm']):
    '''       
    Calculate task Coding metrics:
    
    Two type of metrics are calculated: per document and micro-average.
    It is assumed there are not completely overlapping annotations.
    
    Parameters
    ---------- 
    gs : pandas dataframe
        with the Gold Standard. Columns are those defined in main function.
    pred : pandas dataframe
        with the predictions. Columns are those defined in main function.
    subtask : str
        subtask name
    
    Returns
    -------
    P_per_cc : pandas series
        Precision per clinical case (index contains clinical case names)
    P : float
        Micro-average precision
    R_per_cc : pandas series
        Recall per clinical case (index contains clinical case names)
    R : float
        Micro-average recall
    F1_per_cc : pandas series
        F-score per clinical case (index contains clinical case names)
    F1 : float
        Micro-average F1-score
    '''
    
    # Predicted Positives:
    Pred_Pos_per_cc = \
        pred.drop_duplicates(subset=['filename', "offset"]).\
        groupby("filename")["offset"].count()
    Pred_Pos = pred.drop_duplicates(subset=['filename', "offset"]).shape[0]

    # Gold Standard Positives:
    GS_Pos_per_cc = \
        gs.drop_duplicates(subset=['filename', "offset"]).\
        groupby("filename")["offset"].count()
    GS_Pos = gs.drop_duplicates(subset=['filename', "offset"]).shape[0]
    
    # Eliminate predictions not in GS (prediction needs to be in same clinical
    # case and to have the exact same offset to be considered valid!!!!)
    df_sel = pd.merge(pred, gs, 
                      how="right",
                      on=["filename", "offset", "label"])
    
    if subtask=='norm':
        # Check if codes are equal
        df_sel["is_valid"] = \
            df_sel.apply(lambda x: (x["code_x"] == x["code_y"]), axis=1)
    elif subtask=='ner':
        is_valid = df_sel.apply(lambda x: x.isnull().any()==False, axis=1)
        df_sel = df_sel.assign(is_valid=is_valid.values)
    else:
        raise Exception('Error! Subtask name not properly set up')

        
    # True Positives:
    TP_per_cc = (df_sel[df_sel["is_valid"] == True]
                 .groupby("filename")["is_valid"].count())
    TP = df_sel[df_sel["is_valid"] == True].shape[0]
    
    # Add entries for clinical cases that are not in predictions but are present
    # in the GS
    cc_not_predicted = (pred.drop_duplicates(subset=["filename"])
                        .merge(gs.drop_duplicates(subset=["filename"]), 
                              on='filename',
                              how='right', indicator=True)
                        .query('_merge == "right_only"')
                        .drop('_merge', axis=1))['filename'].to_list()
    for cc in cc_not_predicted:
        TP_per_cc[cc] = 0
    
    # Remove entries for clinical cases that are not in GS but are present
    # in the predictions
    cc_not_GS = (gs.drop_duplicates(subset=["filename"])
                .merge(pred.drop_duplicates(subset=["filename"]), 
                      on='filename',
                      how='right', indicator=True)
                .query('_merge == "right_only"')
                .drop('_merge', axis=1))['filename'].to_list()
    Pred_Pos_per_cc = Pred_Pos_per_cc.drop(cc_not_GS)

    # Calculate Final Metrics:
    P_per_cc =  TP_per_cc / Pred_Pos_per_cc 
    P = TP / Pred_Pos if Pred_Pos > 0 else 0
    R_per_cc = TP_per_cc / GS_Pos_per_cc
    R = TP / GS_Pos if GS_Pos > 0 else 0
    F1_per_cc = (2 * P_per_cc * R_per_cc) / (P_per_cc + R_per_cc)
    if (P+R) == 0:
        F1 = 0
        #warnings.warn('Global F1 score automatically set to zero to avoid division by zero')
        return P_per_cc, P, R_per_cc, R, F1_per_cc, F1
    F1 = (2 * P * R) / (P + R)
    
    if ((any([F1, P, R]) > 1) | any(F1_per_cc>1) | any(P_per_cc>1) | any(R_per_cc>1) ):
        warnings.warn('Metric greater than 1! You have encountered an undetected bug, please, contact antonio.miranda@bsc.es!')
                                            
    return P_per_cc, P, R_per_cc, R, F1_per_cc, F1

In [166]:
def get_occurrences(term, text):
    occurrences = []
    i = 0
    while True:
    	f = text.find(term, i)
    	if f==-1:
    		break
    	occurrences.append(f)
    	i = f+1
    return occurrences

def get_entities(filename, text, term, label):
    entity_list = []
    if term.upper() not in text.upper():
        return entity_list
        
    indices = [(m, m+len(term)) for m in get_occurrences(term.upper(), text.upper())]
    for index in indices:
        start, end = index
        if term.upper() != text.upper()[start:end].upper():
            print(term, text[start:end])
        entity_list.append({
            'filename': filename,
            'mark': 'TOX',
            'label': label,
            'off0': start,
            'off1': end,
            'span': term
        })
    return entity_list

In [167]:
import numpy as np

def get_spans(example, pred):
    filename = example['filename']
    text = example['patient_discharge_summary']
    gold_annotations = example['annotations']
    
    predicted_entities = []
    predicted_spans = []
    for habit in pred.tobacco_habits:
        predicted_entities.append({
            'trigger_type': 'Tobacco',
            'trigger_text': habit
        })

    for habit in pred.alcohol_habits:
        predicted_entities.append({
            'trigger_type': 'Alcohol',
            'trigger_text': habit
        })

    for habit in pred.cannabis_habits:
        predicted_entities.append({
            'trigger_type': 'Cannabis',
            'trigger_text': habit
        })

    for habit in pred.drug_habits:
        predicted_entities.append({
            'trigger_type': 'Drug',
            'trigger_text': habit
        })

    for ent in predicted_entities:
        spans = get_entities(filename, text, ent['trigger_text'], ent['trigger_type'])
        predicted_spans.extend(spans)

    gold_spans = []

    for ent in gold_annotations:
        gold_spans.append({
            'filename': filename,
            'mark': 'TOX',
            'label': ent['label'],
            'off0': ent['off0'],
            'off1': ent['off1'],
            'span': ent['span'],
        })
    return gold_spans, predicted_spans

def f1(example, pred, trace=None): #entities only   
    gold_spans, predicted_spans = get_spans(example, pred)
    
    #  true positives, false positives and false negatives == 0
    #if len(gold_spans) == 0 and len(predicted_spans) == 0:
    #    return 1
    gs = pd.DataFrame.from_records(gold_spans, columns=['filename', 'mark', 'label', 'off0', 'off1', 'span'])
    pred = pd.DataFrame.from_records(predicted_spans, columns=['filename', 'mark', 'label', 'off0', 'off1', 'span'])

    gs['offset'] = gs['off0'].astype(str) + ' ' + gs['off1'].astype(str)
    pred['offset'] = pred['off0'].astype(str) + ' ' + pred['off1'].astype(str)

    # prefer shorter spans, remove overlapping spans?
    gs = gs.sort_values(by=['filename', 'off0', 'off1']).drop_duplicates(subset=['filename', 'label', 'offset']).copy()
    pred = pred.sort_values(by=['filename', 'off0', 'off1']).drop_duplicates(subset=['filename', 'label', 'offset']).copy()
    P_per_cc, P, R_per_cc, R, F1_per_cc, F1 = calculate_metrics(gs, pred, subtask='ner')
       
    return F1

In [ ]:
#evaluate = dspy.Evaluate(devset=dev_dataset_text, metric=f1, num_threads=16, display_progress=True, display_table=5, provide_traceback=True)
#evaluate(doc_extractor)

In [23]:
from dspy.teleprompt import LabeledFewShot

labeled_fewshot_optimizer = LabeledFewShot(k=5)
optimized_fewshot = labeled_fewshot_optimizer.compile(student = doc_extractor, trainset=train_dataset_text)

In [24]:
optimized_fewshot

term_extractor_patient.predict = Predict(StringSignature(patient_discharge_summary -> reasoning, tobacco_habits, cannabis_habits, alcohol_habits, drug_habits
    instructions='You are an expert in clinical NLP in Spanish. \nExtract contiguous tokens referring to specific toxic habits, like substance use and abuse, from a patient discharge summary as they appear in the Spanish text.\nMake sure to retrieve all tokens of toxic habits, if any.'
    patient_discharge_summary = Field(annotation=str required=True json_schema_extra={'desc': 'Patient discharge summary', '__dspy_field_type': 'input', 'prefix': 'Patient Discharge Summary:'})
    reasoning = Field(annotation=str required=True json_schema_extra={'prefix': "Reasoning: Let's think step by step in order to", 'desc': '${reasoning}', '__dspy_field_type': 'output'})
    tobacco_habits = Field(annotation=list[str] required=True json_schema_extra={'desc': 'All tobacco tokens that can be extracted from the discharge summary. For example: ci

In [ ]:
#evaluate(optimized_fewshot) #61.1% - labels 5, 61.39 - 7

In [25]:
optimized_fewshot(patient_discharge_summary=text)

Prediction(
    tobacco_habits=[],
    cannabis_habits=[],
    alcohol_habits=[],
    drug_habits=[]
)

In [26]:
lm.history[-1]

{'prompt': None,
 'messages': [{'role': 'system',
   'content': 'Your input fields are:\n1. `patient_discharge_summary` (str): Patient discharge summary\nYour output fields are:\n1. `reasoning` (str): \n2. `tobacco_habits` (list[str]): All tobacco tokens that can be extracted from the discharge summary. For example: cigarrillos, tabaco, tabaquismo.\n3. `cannabis_habits` (list[str]): All cannabis tokens that can be extracted from the discharge summary. For example: marihuana, cannabis, hachís.\n4. `alcohol_habits` (list[str]): All alcohol tokens that can be extracted from the discharge summary. For example: enolísmo, alcohólica, cervezas.\n5. `drug_habits` (list[str]): All drug tokens that can be extracted from the discharge summary. For example: MDMA, drogas, otros tóxicos, abstinencia, otras sustancias adictivas.\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## patient_discharge_summary ## ]]\n{patient_discharge_summary}\n\n[[ 

In [ ]:
embed_api_key = ""
embed_api_endpoint = "https://<endpoint>.openai.azure.com/"

embedder = dspy.Embedder('azure/text-embedding-3-large', dimensions=512, api_key = embed_api_key, api_base=embed_api_endpoint)

In [ ]:
from dspy import KNNFewShot

knn_labeled_fewshot_optimizer = KNNFewShot(k=5, trainset=train_dataset_text, vectorizer=embedder)
knn_optimized_fewshot = knn_labeled_fewshot_optimizer.compile(doc_extractor)

In [ ]:
evaluate(knn_optimized_fewshot) #50.5% after 34% data

In [168]:
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import scipy.sparse as sp
import typer

label_map = {
    'Tobacco': 1,
    'Alcohol': 2, 
    'Drug': 3, 
    'Cannabis': 4
}

def iou_per_class(user_annotations: pd.DataFrame, target_annotations: pd.DataFrame) -> List[float]:
    """
    Calculate the IoU metric for each class in a set of annotations.
    """
    # Get mapping from note_id to index in array
    docs = np.unique(np.concatenate([user_annotations.filename, target_annotations.filename]))
    doc_index_mapping = dict(zip(docs, range(len(docs))))

    # Identify union of categories in GT and PRED
    cats = [1, 2, 3, 4] #np.unique(np.concatenate([user_annotations.label, target_annotations.label]))

    # Find max character index in GT or PRED
    concat = np.concatenate([user_annotations.off1, target_annotations.off1])
    max_end = np.max(concat) if concat.shape[0] > 0 else 0

    if max_end == 0: # both gold and pred are empty
        return [1] # todo: ignore value

    # Populate matrices for keeping track of character class categorization
    def populate_char_mtx(n_rows, n_cols, annot_df):
        mtx = sp.lil_array((n_rows, n_cols), dtype=np.uint64)
        for row in annot_df.itertuples():
            doc_index = doc_index_mapping[row.filename]
            mtx[doc_index, row.off0 : row.off1] = label_map[row.label]  # noqa: E203
        return mtx.tocsr()

    gt_mtx = populate_char_mtx(docs.shape[0], max_end, target_annotations)
    pred_mtx = populate_char_mtx(docs.shape[0], max_end, user_annotations)

    # Calculate IoU per category
    ious = []
    for cat in cats:
        #try:
        gt_cat = gt_mtx == cat        
        pred_cat = pred_mtx == cat
        # sparse matrices don't support bitwise operators, but the _cat matrices
        # have bool dtypes so when we multiply/add them we end up with only T/F values
        #print(gt_cat.todense())
        #print(pred_cat.todense())
        
        intersection = gt_cat * pred_cat
        union = gt_cat + pred_cat
        #print(intersection)
        #print(union)
        iou = intersection.sum() / union.sum() if union.sum() > 0 else -1
        #except ValueError:
        #    iou = 0.0
        #    pass
        ious.append(iou)

    return ious

In [169]:
def extraction_correctness_metric(example: dspy.Example, prediction: dspy.Prediction, trace=None) -> bool:
    """
    Computes correctness of entity extraction predictions.
    
    Args:
        example (dspy.Example): The dataset example containing expected people entities.
        prediction (dspy.Prediction): The prediction from the DSPy people extraction program.
        trace: Optional trace object for debugging.
    
    Returns:
        bool: True if predictions match expectations, False otherwise.
    """
    #print(example)
    #print(prediction)
    gold_spans, predicted_spans = get_spans(example, prediction)    
    gs = pd.DataFrame.from_records(gold_spans, columns=['filename', 'mark', 'label', 'off0', 'off1', 'span'])
    pred = pd.DataFrame.from_records(predicted_spans, columns=['filename', 'mark', 'label', 'off0', 'off1', 'span'])
    #print(gs)
    #print(pred)
    ious = iou_per_class(pred, gs)
    #print(ious)
    ious = [iou for iou in ious if iou >= 0] # remove categories which are not present in neither gold or pred
    #print(ious)
    return np.mean(ious) #if len(ious) > 0 else 0

In [170]:
evaluate_correctness = dspy.Evaluate(
    devset=dev_dataset_text_labels,
    metric=extraction_correctness_metric,
    num_threads=24,
    display_progress=True,
    display_table=True
)

In [172]:
#    term_extractor_patient, #doc_extractor
evaluate_correctness(term_extractor_patient, devset=dev_dataset_text_labels)

Average Metric: 293.42 / 520 (56.4%): 100%|██████████████████████████████| 520/520 [00:03<00:00, 172.85it/s]

2025/06/15 14:24:58 INFO dspy.evaluate.evaluate: Average Metric: 293.4209601786316 / 520 (56.4%)


,patient_discharge_summary,filename,annotations,example_tobacco_habits,example_cannabis_habits,example_alcohol_habits,example_drug_habits,reasoning,pred_tobacco_habits,pred_cannabis_habits,pred_alcohol_habits,pred_drug_habits,extraction_correctness_metric
0,﻿Un hombre de 36 años llegó al servicio de urgencias con presunta ...,32423911_ES_0,"[{'filename': '32423911_ES', 'mark': 'TOX', 'label': 'Tobacco', 'o...",[fumador],[],[],[],"En el resumen de alta, se menciona explícitamente que el paciente ...","[fumador, cajetillas-año]",[],[],[],✔️ [0.333]
1,"Anamnesis Sexo masculino, 79 años. Autoválido. Procedente de Salto...",casos_clinicos_cardiologia10_0,"[{'filename': 'casos_clinicos_cardiologia10', 'mark': 'TOX', 'labe...",[Ex-tabaquista],[],[],[],"En el resumen de alta, se menciona explícitamente ""Ex-tabaquista"" ...",[Ex-tabaquista],[],[],[],✔️ [1.000]
2,Varón de 61 años de edad que acude en junio del 2010 a la consulta...,casos_clinicos_cardiologia31_0,"[{'filename': 'casos_clinicos_cardiologia31', 'mark': 'TOX', 'labe...",[exfumador],[],[],[],"En el resumen de alta se menciona ""exfumador"" como parte de los an...",[exfumador],[],[],[],✔️ [1.000]
3,ANTECEDENTES: - No reacciones alérgicas medicamentosas conocidas. ...,casos_clinicos_cardiologia4_19,"[{'filename': 'casos_clinicos_cardiologia4', 'mark': 'TOX', 'label...",[exfumador],[],[],[],"En la sección de antecedentes, bajo ""Hábitos tóxicos"", se menciona...",[exfumador],[],[],[],✔️ [1.000]
4,"ANTECEDENTES, ENFERMEDAD ACTUAL Y EXPLORACIÓN FÍSICA Hombre de 68 ...",casos_clinicos_cardiologia64_0,"[{'filename': 'casos_clinicos_cardiologia64', 'mark': 'TOX', 'labe...",[exfumador],[],[],[],"En el resumen de alta se menciona explícitamente ""exfumador"", lo q...",[exfumador],[],[],[],✔️ [1.000]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,"El exceso percibido de consumo queda reflejado en la figura 5, que...",S1130-52742017000100001-1_16140,"[{'filename': 'S1130-52742017000100001-1', 'mark': 'TOX', 'label':...",[],[],"[alcohol, bebidas alcohólicas, beber]","[consumo, consumo]","En el resumen de alta se menciona explícitamente el ""consumo de al...",[],[],"[consumo de alcohol, ingesta de bebidas alcohólicas]",[],✔️ [0.245]
516,En la figura 3 podemos observar el cumplimiento de actividades de ...,S1130-52742017000100001-1_16732,"[{'filename': 'S1130-52742017000100001-1', 'mark': 'TOX', 'label':...",[],[],[alcohol],[],"En el resumen de alta se menciona explícitamente el ""consumo de al...",[],[],[alcohol],[],✔️ [1.000]
517,"Paciente de 51 años, fumadora, sin alergias medicamentosas conocid...",S1130-63432015000600011-1_0,"[{'filename': 'S1130-63432015000600011-1', 'mark': 'TOX', 'label':...",[fumadora],[],[],[],En el resumen de alta se menciona explícitamente que la paciente e...,[fumadora],[],[],[],✔️ [1.000]
518,Lactante de sexo femenino que ingresó a los 7 meses de vida por pr...,S1137-66272009000500017-1_0,"[{'filename': 'S1137-66272009000500017-1', 'mark': 'TOX', 'label':...",[],[cannabis],[],"[cocaína, cocaína]",En el resumen de alta se mencionan explícitamente varias sustancia...,[],[cannabis],[],"[tóxicos, cocaína]",✔️ [0.750]


56.43

In [178]:
#    term_extractor_patient, #doc_extractor
evaluate_correctness(term_extractor_patient, devset=dev_dataset_text)

Average Metric: 1440.42 / 1735 (83.0%): 100%|██████████████████████████| 1735/1735 [00:06<00:00, 253.43it/s]

2025/06/15 17:18:59 INFO dspy.evaluate.evaluate: Average Metric: 1440.420960178633 / 1735 (83.0%)


,patient_discharge_summary,filename,annotations,example_tobacco_habits,example_cannabis_habits,example_alcohol_habits,example_drug_habits,reasoning,pred_tobacco_habits,pred_cannabis_habits,pred_alcohol_habits,pred_drug_habits,extraction_correctness_metric
0,﻿Un hombre de 36 años llegó al servicio de urgencias con presunta ...,32423911_ES_0,"[{'filename': '32423911_ES', 'mark': 'TOX', 'label': 'Tobacco', 'o...",[fumador],[],[],[],"En el resumen de alta, se menciona explícitamente que el paciente ...","[fumador, cajetillas-año]",[],[],[],✔️ [0.333]
1,"Anamnesis Sexo masculino, 79 años. Autoválido. Procedente de Salto...",casos_clinicos_cardiologia10_0,"[{'filename': 'casos_clinicos_cardiologia10', 'mark': 'TOX', 'labe...",[Ex-tabaquista],[],[],[],"En el resumen de alta, se menciona explícitamente ""Ex-tabaquista"" ...",[Ex-tabaquista],[],[],[],✔️ [1.000]
2,"15/11/16 Instala síndrome confusional, fiebre 40o C axilar y disne...",casos_clinicos_cardiologia10_785,[],[],[],[],[],The discharge summary describes an episode of confusional syndrome...,[],[],[],[],✔️ [1.000]
3,"Al examen se destaca: GCS 12, FR 26 rpm, PA 100/60mmHg, Ta x 38,5o...",casos_clinicos_cardiologia10_888,[],[],[],[],[],He revisado cuidadosamente el resumen de alta proporcionado y no s...,[],[],[],[],✔️ [1.000]
4,Exámenes diagnósticos relevantes • RxTx Redistribución de flujo a ...,casos_clinicos_cardiologia10_1380,[],[],[],[],[],The discharge summary provided is focused on diagnostic test resul...,[],[],[],[],✔️ [1.000]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1730,,S1137-66272009000500017-1_0,[],[],[],[],[],No se proporciona ningún texto en el campo de resumen de alta del ...,[],[],[],[],✔️ [1.000]
1731,"Paciente de 58 años, con historia de cefalea recurrente, motivo po...",S1699-695X2015000200009-1_0,"[{'filename': 'S1699-695X2015000200009-1', 'mark': 'TOX', 'label':...",[],[],[],[tóxicos],"En el resumen de alta se menciona explícitamente ""Niega consumo de...",[],[],[],[],
1732,En el Servicio de Urgencias se pautó gastroprotección y analgésico...,S1699-695X2015000200009-1_1516,[],[],[],[],[],He revisado cuidadosamente el resumen de alta proporcionado. El te...,[],[],[],[],✔️ [1.000]
1733,• Doppler orbitario: se objetivan signos de hipertensión intracran...,S1699-695X2015000200009-1_2219,[],[],[],[],[],"The discharge summary focuses on diagnostic imaging, serology, and...",[],[],[],[],✔️ [1.000]


83.02

In [173]:
mipro_optimizer = dspy.MIPROv2(
    metric=extraction_correctness_metric,
    auto="medium",
)
optimized_patient_extractor = mipro_optimizer.compile(
    doc_extractor, #term_extractor_patient
    trainset=train_dataset_text,
    max_bootstrapped_demos=4,
    requires_permission_to_run=False,
    minibatch=False
)

2025/06/15 14:25:06 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING MEDIUM AUTO RUN SETTINGS:
num_trials: 18
minibatch: True
num_fewshot_candidates: 12
num_instruct_candidates: 6
valset size: 300

2025/06/15 14:25:06 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/06/15 14:25:06 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/06/15 14:25:06 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=12 sets of demonstrations...


Bootstrapping set 1/12
Bootstrapping set 2/12
Bootstrapping set 3/12


  1%|▌                                                                      | 6/692 [00:08<16:44,  1.46s/it]


Bootstrapped 4 full traces after 6 examples for up to 1 rounds, amounting to 6 attempts.
Bootstrapping set 4/12


  0%|                                                                       | 1/692 [00:02<31:10,  2.71s/it]


...


Average Metric: 22.28 / 35 (63.6%): 100%|███████████████████████████████████| 35/35 [01:02<00:00,  1.79s/it]

2025/06/15 15:55:06 INFO dspy.evaluate.evaluate: Average Metric: 22.2762366717125 / 35 (63.6%)
2025/06/15 15:55:06 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 63.65 on minibatch of size 35 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 9'].
2025/06/15 15:55:06 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [42.7, 41.71, 48.11, 53.62, 52.4, 42.03, 36.45, 51.61, 39.96, 49.93, 67.78, 59.98, 52.26, 53.52, 39.13, 50.74, 61.9, 63.65]
2025/06/15 15:55:06 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [53.41, 55.05, 51.01, 57.66]
2025/06/15 15:55:06 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 57.66
2025/06/15 15:55:06 INFO dspy.teleprompt.mipro_optimizer_v2: ==========================================


2025/06/15 15:55:06 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 23 / 23 - Full Evaluation =====
2025/06/15 15:55:06 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging pr


Average Metric: 184.62 / 300 (61.5%): 100%|███████████████████████████████| 300/300 [10:19<00:00,  2.06s/it]

2025/06/15 16:05:25 INFO dspy.evaluate.evaluate: Average Metric: 184.617675847156 / 300 (61.5%)
2025/06/15 16:05:25 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 61.54


2025/06/15 16:05:25 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [53.41, 55.05, 51.01, 57.66, 61.54]
2025/06/15 16:05:25 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 61.54
2025/06/15 16:05:25 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/06/15 16:05:25 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/06/15 16:05:25 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 61.54!


In [174]:
optimized_patient_extractor.save("optimized_extractor_v2.json")

In [175]:
evaluate_correctness(optimized_patient_extractor, devset=dev_dataset_text_labels)

Average Metric: 344.85 / 520 (66.3%): 100%|███████████████████████████████| 520/520 [21:24<00:00,  2.47s/it]

2025/06/15 16:26:50 INFO dspy.evaluate.evaluate: Average Metric: 344.84613411798904 / 520 (66.3%)


,patient_discharge_summary,filename,annotations,example_tobacco_habits,example_cannabis_habits,example_alcohol_habits,example_drug_habits,pred_tobacco_habits,pred_cannabis_habits,pred_alcohol_habits,pred_drug_habits,extraction_correctness_metric
0,﻿Un hombre de 36 años llegó al servicio de urgencias con presunta ...,32423911_ES_0,"[{'filename': '32423911_ES', 'mark': 'TOX', 'label': 'Tobacco', 'o...",[fumador],[],[],[],[fumador],[],[],[],✔️ [1.000]
1,"Anamnesis Sexo masculino, 79 años. Autoválido. Procedente de Salto...",casos_clinicos_cardiologia10_0,"[{'filename': 'casos_clinicos_cardiologia10', 'mark': 'TOX', 'labe...",[Ex-tabaquista],[],[],[],[Ex-tabaquista],[],[],[],✔️ [1.000]
2,Varón de 61 años de edad que acude en junio del 2010 a la consulta...,casos_clinicos_cardiologia31_0,"[{'filename': 'casos_clinicos_cardiologia31', 'mark': 'TOX', 'labe...",[exfumador],[],[],[],[exfumador],[],[],[],✔️ [1.000]
3,ANTECEDENTES: - No reacciones alérgicas medicamentosas conocidas. ...,casos_clinicos_cardiologia4_19,"[{'filename': 'casos_clinicos_cardiologia4', 'mark': 'TOX', 'label...",[exfumador],[],[],[],[exfumador],[],[],[],✔️ [1.000]
4,"ANTECEDENTES, ENFERMEDAD ACTUAL Y EXPLORACIÓN FÍSICA Hombre de 68 ...",casos_clinicos_cardiologia64_0,"[{'filename': 'casos_clinicos_cardiologia64', 'mark': 'TOX', 'labe...",[exfumador],[],[],[],[exfumador],[],[],[],✔️ [1.000]
...,...,...,...,...,...,...,...,...,...,...,...,...
515,"El exceso percibido de consumo queda reflejado en la figura 5, que...",S1130-52742017000100001-1_16140,"[{'filename': 'S1130-52742017000100001-1', 'mark': 'TOX', 'label':...",[],[],"[alcohol, bebidas alcohólicas, beber]","[consumo, consumo]",[],[],"[consumo de alcohol, ingesta de bebidas alcohólicas, beber]",[],✔️ [0.292]
516,En la figura 3 podemos observar el cumplimiento de actividades de ...,S1130-52742017000100001-1_16732,"[{'filename': 'S1130-52742017000100001-1', 'mark': 'TOX', 'label':...",[],[],[alcohol],[],[],[],[alcohol],[],✔️ [1.000]
517,"Paciente de 51 años, fumadora, sin alergias medicamentosas conocid...",S1130-63432015000600011-1_0,"[{'filename': 'S1130-63432015000600011-1', 'mark': 'TOX', 'label':...",[fumadora],[],[],[],[fumadora],[],[],[],✔️ [1.000]
518,Lactante de sexo femenino que ingresó a los 7 meses de vida por pr...,S1137-66272009000500017-1_0,"[{'filename': 'S1137-66272009000500017-1', 'mark': 'TOX', 'label':...",[],[cannabis],[],"[cocaína, cocaína]",[],[cannabis],[],"[cocaína, metabolitos de cocaína]",✔️ [0.694]


66.32

In [176]:
evaluate_correctness(optimized_patient_extractor, devset=dev_dataset_text)

Average Metric: 1489.65 / 1735 (85.9%): 100%|███████████████████████████| 1735/1735 [44:58<00:00,  1.56s/it]

2025/06/15 17:17:50 INFO dspy.evaluate.evaluate: Average Metric: 1489.6516896735454 / 1735 (85.9%)


,patient_discharge_summary,filename,annotations,example_tobacco_habits,example_cannabis_habits,example_alcohol_habits,example_drug_habits,pred_tobacco_habits,pred_cannabis_habits,pred_alcohol_habits,pred_drug_habits,extraction_correctness_metric
0,﻿Un hombre de 36 años llegó al servicio de urgencias con presunta ...,32423911_ES_0,"[{'filename': '32423911_ES', 'mark': 'TOX', 'label': 'Tobacco', 'o...",[fumador],[],[],[],[fumador],[],[],[],✔️ [1.000]
1,"Anamnesis Sexo masculino, 79 años. Autoválido. Procedente de Salto...",casos_clinicos_cardiologia10_0,"[{'filename': 'casos_clinicos_cardiologia10', 'mark': 'TOX', 'labe...",[Ex-tabaquista],[],[],[],[Ex-tabaquista],[],[],[],✔️ [1.000]
2,"15/11/16 Instala síndrome confusional, fiebre 40o C axilar y disne...",casos_clinicos_cardiologia10_785,[],[],[],[],[],[],[],[],[],✔️ [1.000]
3,"Al examen se destaca: GCS 12, FR 26 rpm, PA 100/60mmHg, Ta x 38,5o...",casos_clinicos_cardiologia10_888,[],[],[],[],[],[],[],[],[],✔️ [1.000]
4,Exámenes diagnósticos relevantes • RxTx Redistribución de flujo a ...,casos_clinicos_cardiologia10_1380,[],[],[],[],[],[],[],[],[],✔️ [1.000]
...,...,...,...,...,...,...,...,...,...,...,...,...
1730,,S1137-66272009000500017-1_0,[],[],[],[],[],[],[],[],[],✔️ [1.000]
1731,"Paciente de 58 años, con historia de cefalea recurrente, motivo po...",S1699-695X2015000200009-1_0,"[{'filename': 'S1699-695X2015000200009-1', 'mark': 'TOX', 'label':...",[],[],[],[tóxicos],[],[],[],[],
1732,En el Servicio de Urgencias se pautó gastroprotección y analgésico...,S1699-695X2015000200009-1_1516,[],[],[],[],[],[],[],[],[],✔️ [1.000]
1733,• Doppler orbitario: se objetivan signos de hipertensión intracran...,S1699-695X2015000200009-1_2219,[],[],[],[],[],[],[],[],[],✔️ [1.000]


85.86

In [177]:
lm.history[-1]

{'prompt': None,
 'messages': [{'role': 'system',
   'content': 'Your input fields are:\n1. `patient_discharge_summary` (str): Patient discharge summary\nYour output fields are:\n1. `reasoning` (str): \n2. `tobacco_habits` (list[str]): All tobacco tokens that can be extracted from the discharge summary. For example: cigarrillos, tabaco, tabaquismo.\n3. `cannabis_habits` (list[str]): All cannabis tokens that can be extracted from the discharge summary. For example: marihuana, cannabis, hachís.\n4. `alcohol_habits` (list[str]): All alcohol tokens that can be extracted from the discharge summary. For example: enolísmo, alcohólica, cervezas.\n5. `drug_habits` (list[str]): All drug tokens that can be extracted from the discharge summary. For example: MDMA, drogas, otros tóxicos, abstinencia, otras sustancias adictivas.\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## patient_discharge_summary ## ]]\n{patient_discharge_summary}\n\n[[ 

In [180]:
loaded_patient_extractor = SplitExtract() #dspy.ChainOfThought(PatientRecord)
loaded_patient_extractor.load("optimized_extractor_v2.json")

In [181]:
text = dev_dataset_text[0].patient_discharge_summary
loaded_patient_extractor(patient_discharge_summary=text)

Prediction(
    tobacco_habits=['fumador'],
    cannabis_habits=[],
    alcohol_habits=[],
    drug_habits=[]
)

In [182]:
evaluate_correctness(loaded_patient_extractor, devset=dev_dataset_text)

Average Metric: 1489.65 / 1735 (85.9%): 100%|██████████████████████████| 1735/1735 [00:07<00:00, 219.57it/s]


2025/06/15 17:22:20 INFO dspy.evaluate.evaluate: Average Metric: 1489.6516896735454 / 1735 (85.9%)


,patient_discharge_summary,filename,annotations,example_tobacco_habits,example_cannabis_habits,example_alcohol_habits,example_drug_habits,pred_tobacco_habits,pred_cannabis_habits,pred_alcohol_habits,pred_drug_habits,extraction_correctness_metric
0,﻿Un hombre de 36 años llegó al servicio de urgencias con presunta ...,32423911_ES_0,"[{'filename': '32423911_ES', 'mark': 'TOX', 'label': 'Tobacco', 'o...",[fumador],[],[],[],[fumador],[],[],[],✔️ [1.000]
1,"Anamnesis Sexo masculino, 79 años. Autoválido. Procedente de Salto...",casos_clinicos_cardiologia10_0,"[{'filename': 'casos_clinicos_cardiologia10', 'mark': 'TOX', 'labe...",[Ex-tabaquista],[],[],[],[Ex-tabaquista],[],[],[],✔️ [1.000]
2,"15/11/16 Instala síndrome confusional, fiebre 40o C axilar y disne...",casos_clinicos_cardiologia10_785,[],[],[],[],[],[],[],[],[],✔️ [1.000]
3,"Al examen se destaca: GCS 12, FR 26 rpm, PA 100/60mmHg, Ta x 38,5o...",casos_clinicos_cardiologia10_888,[],[],[],[],[],[],[],[],[],✔️ [1.000]
4,Exámenes diagnósticos relevantes • RxTx Redistribución de flujo a ...,casos_clinicos_cardiologia10_1380,[],[],[],[],[],[],[],[],[],✔️ [1.000]
...,...,...,...,...,...,...,...,...,...,...,...,...
1730,,S1137-66272009000500017-1_0,[],[],[],[],[],[],[],[],[],✔️ [1.000]
1731,"Paciente de 58 años, con historia de cefalea recurrente, motivo po...",S1699-695X2015000200009-1_0,"[{'filename': 'S1699-695X2015000200009-1', 'mark': 'TOX', 'label':...",[],[],[],[tóxicos],[],[],[],[],
1732,En el Servicio de Urgencias se pautó gastroprotección y analgésico...,S1699-695X2015000200009-1_1516,[],[],[],[],[],[],[],[],[],✔️ [1.000]
1733,• Doppler orbitario: se objetivan signos de hipertensión intracran...,S1699-695X2015000200009-1_2219,[],[],[],[],[],[],[],[],[],✔️ [1.000]


85.86

In [183]:
df_records_dev.head()

,filename,text,trigger_annotations,attr_annotations,is_train,response,entities
2,32423911_ES,﻿Un hombre de 36 años llegó al servicio de urgencias con presunta ...,"[{'label': 'Tobacco', 'off0': '350', 'off1': '357', 'span': 'fumad...","[{'label': 'Amount', 'off0': '361', 'off1': '378', 'span': '10 caj...",False,,
5,casos_clinicos_cardiologia10,"Anamnesis\nSexo masculino, 79 años. Autoválido. Procedente de Salt...","[{'label': 'Tobacco', 'off0': '126', 'off1': '139', 'span': 'Ex-ta...",[],False,,
16,casos_clinicos_cardiologia31,Varón de 61 años de edad que acude en junio del 2010 a la consulta...,"[{'label': 'Tobacco', 'off0': '335', 'off1': '344', 'span': 'exfum...",[],False,,
17,casos_clinicos_cardiologia4,Varón de 70 años.\n\nANTECEDENTES:\n- No reacciones alérgicas medi...,"[{'label': 'Tobacco', 'off0': '104', 'off1': '113', 'span': 'exfum...",[],False,,
19,casos_clinicos_cardiologia64,"ANTECEDENTES, ENFERMEDAD ACTUAL Y EXPLORACIÓN FÍSICA\nHombre de 68...","[{'label': 'Tobacco', 'off0': '123', 'off1': '132', 'span': 'exfum...",[],False,,


In [184]:
from tqdm import tqdm, tqdm_notebook

for index, row in tqdm(df_records_dev.iterrows(), total=df_records_dev.shape[0]):
    if row['response'] != '':
        continue
    responses = []
    entities = []
    texts = [row['text']] #.split('\n\n') #[row['text']] # 
    for text in texts:
        #response = optimized_fewshot(patient_discharge_summary=text) 
        #response = knn_optimized_fewshot(patient_discharge_summary=text) 
        #response = optimized_fewshot(patient_discharge_summary=text) 
        response = loaded_patient_extractor(patient_discharge_summary=text) 
        responses.append(response)
        current_entities = []
        for habit in response.tobacco_habits:
            current_entities.append({
                'trigger_type': 'Tobacco',
                'trigger_text': habit
            })
        for habit in response.alcohol_habits:
            current_entities.append({
                'trigger_type': 'Alcohol',
                'trigger_text': habit
            })
        for habit in response.cannabis_habits:
            current_entities.append({
                'trigger_type': 'Cannabis',
                'trigger_text': habit
            })
        for habit in response.drug_habits:
            current_entities.append({
                'trigger_type': 'Drug',
                'trigger_text': habit
            })
        entities.extend(current_entities)
    df_records_dev.at[index, 'response'] = responses
    df_records_dev.at[index, 'entities'] = entities

100%|█████████████████████████████████████████████████████████████████████| 300/300 [00:03<00:00, 84.33it/s]


In [185]:
df_records_dev.head()

,filename,text,trigger_annotations,attr_annotations,is_train,response,entities
2,32423911_ES,﻿Un hombre de 36 años llegó al servicio de urgencias con presunta ...,"[{'label': 'Tobacco', 'off0': '350', 'off1': '357', 'span': 'fumad...","[{'label': 'Amount', 'off0': '361', 'off1': '378', 'span': '10 caj...",False,"[[tobacco_habits, cannabis_habits, alcohol_habits, drug_habits]]","[{'trigger_type': 'Tobacco', 'trigger_text': 'fumador'}]"
5,casos_clinicos_cardiologia10,"Anamnesis\nSexo masculino, 79 años. Autoválido. Procedente de Salt...","[{'label': 'Tobacco', 'off0': '126', 'off1': '139', 'span': 'Ex-ta...",[],False,"[[tobacco_habits, cannabis_habits, alcohol_habits, drug_habits]]","[{'trigger_type': 'Tobacco', 'trigger_text': 'Ex-tabaquista'}]"
16,casos_clinicos_cardiologia31,Varón de 61 años de edad que acude en junio del 2010 a la consulta...,"[{'label': 'Tobacco', 'off0': '335', 'off1': '344', 'span': 'exfum...",[],False,"[[tobacco_habits, cannabis_habits, alcohol_habits, drug_habits]]","[{'trigger_type': 'Tobacco', 'trigger_text': 'exfumador'}]"
17,casos_clinicos_cardiologia4,Varón de 70 años.\n\nANTECEDENTES:\n- No reacciones alérgicas medi...,"[{'label': 'Tobacco', 'off0': '104', 'off1': '113', 'span': 'exfum...",[],False,"[[tobacco_habits, cannabis_habits, alcohol_habits, drug_habits]]","[{'trigger_type': 'Tobacco', 'trigger_text': 'exfumador'}]"
19,casos_clinicos_cardiologia64,"ANTECEDENTES, ENFERMEDAD ACTUAL Y EXPLORACIÓN FÍSICA\nHombre de 68...","[{'label': 'Tobacco', 'off0': '123', 'off1': '132', 'span': 'exfum...",[],False,"[[tobacco_habits, cannabis_habits, alcohol_habits, drug_habits]]","[{'trigger_type': 'Tobacco', 'trigger_text': 'exfumador'}]"


In [186]:
df_records_dev.iloc[1]['entities']

[{'trigger_type': 'Tobacco', 'trigger_text': 'Ex-tabaquista'}]

In [187]:
result_filename

'pred_dev_gpt41_lists_examples_per_field_temp0_optimizers'

In [188]:
df_records_dev.to_csv(f'{result_filename}.tsv', sep='\t', index=False)

In [189]:
def get_occurrences(term, text):
    occurrences = []
    i = 0
    while True:
    	f = text.find(term, i)
    	if f==-1:
    		break
    	occurrences.append(f)
    	i = f+1
    return occurrences

def get_entities(filename, text, term, label):
    entity_list = []
    if term.upper() not in text.upper():
        return entity_list
        
    indices = [(m, m+len(term)) for m in get_occurrences(term.upper(), text.upper())]
    for index in indices:
        start, end = index
        if term.upper() != text.upper()[start:end].upper():
            print(term, text[start:end])
        entity_list.append({
            'filename': filename,
            'mark': 'TOX',
            'label': label,
            'off0': start,
            'off1': end,
            'span': term
        })
    return entity_list

In [190]:
import re

entities_list = []

for index, row in df_records_dev.iterrows():
    if row['response'] == '':
        continue
        
    entities = row['entities']
    text = row['text']
    
    entity_list = []
    for ent in entities:
        term = ent[f'trigger_text']
        label = ent['trigger_type']

        entity_list.extend(get_entities(row['filename'], text, term, label))        
        
    entities_list.extend(entity_list)    

In [191]:
import pandas as pd

df_entities_list = pd.DataFrame.from_records(entities_list)
df_entities_list.drop_duplicates(inplace=True)
df_entities_list.head()

,filename,mark,label,off0,off1,span
0,32423911_ES,TOX,Tobacco,350,357,fumador
1,casos_clinicos_cardiologia10,TOX,Tobacco,126,139,Ex-tabaquista
2,casos_clinicos_cardiologia31,TOX,Tobacco,335,344,exfumador
3,casos_clinicos_cardiologia4,TOX,Tobacco,104,113,exfumador
4,casos_clinicos_cardiologia64,TOX,Tobacco,123,132,exfumador


In [192]:
#filename, label, off0, off1, span
df_entities_list[['filename','label','off0','off1','span']].to_csv(f'{result_filename}_entities.tsv', sep='\t', index=False)
df_entities_list.shape

(2462, 6)

In [193]:
result_filename

'pred_dev_gpt41_lists_examples_per_field_temp0_optimizers'